In [1]:
# ── ЯЧЕЙКА 1. SETUP ─────────────────────────────────────────────────────
!pip -q install --upgrade diffusers>=0.29 transformers>=4.40 accelerate safetensors \
                          sentencepiece opencv-python-headless pandas numpy pillow tqdm
!pip -q install --upgrade "Pillow>=11.0,<12"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
dopamine-rl 4.1.2 r

In [2]:
import os, gc, re, shutil, zipfile, urllib.request
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch
from PIL import Image, ImageDraw, ImageFont, ImageEnhance, ImageFilter
from tqdm.auto import tqdm

WORK = Path('/kaggle/working'); WORK.mkdir(parents=True, exist_ok=True)
print('Working dir:', WORK, '| CUDA:', torch.cuda.is_available())

# HF-токен из Kaggle Secrets (Add-ons → Secrets, ключ HF_TOKEN)
HF_TOKEN='YOUR_HF_TOKEN'
# try:
#     from kaggle_secrets import UserSecretsClient
#     HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
# except Exception as e:
#     print('Secrets не подхватились:', e)
# Не рекомендуется, но если Secrets не настроены — можно прописать вручную:
HF_TOKEN='YOUR_HF_TOKEN'
assert HF_TOKEN, 'Нужен HF_TOKEN (Kaggle Secrets) + принятая лицензия SD3 Medium на HuggingFace.' 

Working dir: /kaggle/working | CUDA: True


In [3]:
# ── ЯЧЕЙКА 2. КОРПУС: CSV → чистка → руны ──────────────────────────────
# Укажите имя вашего Kaggle Dataset с CSV-файлами.
# После Add data они окажутся в /kaggle/input/<dataset-slug>/.
INPUT_DIR = Path('/kaggle/input')          # сканируем подключённые датасеты
CSV_NAMES = ['datasets/zhopa228/christerhamp-gamla-runor-with-filenames/christerhamp_gamla_runor_with_filenames.csv',
             'datasets/zhopa228/runic-inscriptions/runic_inscriptions_20251228_140136.csv',
            'datasets/zhopa228/runer-ku/runer_ku_20260114_234216.csv']
def find_csvs():
    found = {}
    for name in CSV_NAMES:
        for p in INPUT_DIR.rglob(name):
            found[name] = p; break
    if len(found) < len(CSV_NAMES):
        # fallback: возможно, CSV лежат в /kaggle/working (например, загружены вручную)
        for name in CSV_NAMES:
            p = WORK/name
            if p.exists(): found[name] = p
    missing = [n for n in CSV_NAMES if n not in found]
    assert not missing, f'Не найдены CSV: {missing}. Подключите датасет (Add data) или положите файлы в /kaggle/working.'
    return [found[n] for n in CSV_NAMES]

CSV_PATHS = find_csvs()
print('CSV входы:'); [print(' ', p) for p in CSV_PATHS]

TRANSLIT_TO_RUNE={
    "f":"\u16A0","u":"\u16A2","þ":"\u16A6","ą":"\u16AC","r":"\u16B1","k":"\u16B4",
    "h":"\u16BC","n":"\u16BE","i":"\u16C1","a":"\u16C5","s":"\u16CB","t":"\u16CF",
    "b":"\u16D2","m":"\u16D8","l":"\u16DA","R":"\u16E6","e":"\u16C2","g":"\u16B5",
    "d":"\u16D1","p":"\u16D4","y":"\u16A4","o":"\u16AE","c":"\u16CD","z":"\u16CE",
    "v":"\u16A1","ʀ":"\u16E7","w":"\u16B9","j":"\u16C3","ð":"\u16A7","ï":"\u16C7",
    "ŋ":"\u16DC","æ":"\u16AB","ø":"\u16AF"}
RUNE_TO_TRANSLIT={v:k for k,v in TRANSLIT_TO_RUNE.items()}
VALID=set(TRANSLIT_TO_RUNE); _SPLIT=re.compile("[^"+re.escape("".join(VALID))+"]+")
def tokenize_words(s,min_len=3,max_len=25):
    if not isinstance(s,str) or not s: return []
    s=s.replace("A","a").replace("\u1D00","a"); out=[]
    for tok in _SPLIT.split(s):
        if tok and min_len<=len(tok)<=max_len and all(c in VALID for c in tok): out.append(tok)
    return out
def word_to_runes(w): return "".join(TRANSLIT_TO_RUNE[c] for c in w)
def build_corpus(paths,col='transliteration',min_len=3,max_len=25,min_count=2):
    from collections import Counter; cnt=Counter()
    for p in paths:
        df=pd.read_csv(p)
        for s in df[col].dropna().astype(str): cnt.update(tokenize_words(s,min_len,max_len))
    rows=[{'word':w,'runic':word_to_runes(w),'count':c,'length':len(w)}
          for w,c in cnt.items() if c>=min_count]
    return pd.DataFrame(rows).sort_values('count',ascending=False).reset_index(drop=True)
corpus=build_corpus(CSV_PATHS,min_len=3,max_len=25,min_count=2)
assert all("".join(RUNE_TO_TRANSLIT[c] for c in r)==w for w,r in zip(corpus.word,corpus.runic))
corpus.to_csv(WORK/'runic_corpus.csv',index=False,encoding='utf-8')
print(f'Корпус: {len(corpus)} слов (round-trip OK)'); print(corpus.head(8).to_string(index=False))

CSV входы:
  /kaggle/input/datasets/zhopa228/christerhamp-gamla-runor-with-filenames/christerhamp_gamla_runor_with_filenames.csv
  /kaggle/input/datasets/zhopa228/runic-inscriptions/runic_inscriptions_20251228_140136.csv
  /kaggle/input/datasets/zhopa228/runer-ku/runer_ku_20260114_234216.csv
Корпус: 8562 слов (round-trip OK)
 word runic  count  length
  sin   ᛋᛁᚾ   2586       3
  auk   ᛅᚢᚴ   2149       3
faþur ᚠᛅᚦᚢᚱ   1094       5
stain ᛋᛏᛅᛁᚾ    952       5
 stin  ᛋᛏᛁᚾ    914       4
  sun   ᛋᚢᚾ    845       3
  lit   ᛚᛁᛏ    799       3
  ide   ᛁᛑᛂ    653       3


In [4]:
# ── ЯЧЕЙКА 3. СЭМПЛЕР + РЕНДЕР (авто-кегль, перенос) + Canny ────────────
DIVIDER_RUNE="\u16EC"   # ᛬
class CorpusSampler:
    def __init__(self,df,temperature=0.7):
        self.words=df.word.to_numpy(); self.runes=df.runic.to_numpy()
        c=df['count'].to_numpy().astype(float)
        w=np.ones_like(c) if temperature<=0 else np.power(c,temperature); self.p=w/w.sum()
    def word(self,rng):
        i=int(rng.choice(len(self.words),p=self.p)); return str(self.runes[i]),str(self.words[i])
    def phrase(self,rng,min_words=1,max_words=4):
        n=int(rng.integers(min_words,max_words+1)); R,T=[],[]
        for _ in range(n):
            r,t=self.word(rng); R.append(r); T.append(t)
        return (DIVIDER_RUNE.join(R) if n>1 else R[0]), " ".join(T)
_FONT_URL=("https://github.com/notofonts/NotoSansRunic/raw/refs/heads/main"
           "/fonts/ttf/unhinted/instance_ttf/NotoSansRunic-Regular.ttf")
FONT_PATH=str(WORK/'NotoSansRunic-Regular.ttf')
if not Path(FONT_PATH).exists(): urllib.request.urlretrieve(_FONT_URL,FONT_PATH)
def _layout(draw,text,req,canvas,margin,floor=28):
    usable=canvas-2*margin
    for fs in range(req,floor-1,-4):
        f=ImageFont.truetype(FONT_PATH,fs)
        if draw.textbbox((0,0),text,font=f)[2]<=usable: return f,[text]
    f=ImageFont.truetype(FONT_PATH,floor); lines,cur=[], ''
    for wd in text.split(DIVIDER_RUNE):
        trial=(cur+DIVIDER_RUNE+wd) if cur else wd
        if draw.textbbox((0,0),trial,font=f)[2]<=usable: cur=trial
        else:
            if cur: lines.append(cur)
            cur=wd
    if cur: lines.append(cur)
    return f,lines
def render_runes(text,canvas=1024,req_size=140,margin=44,rng=None):
    base=Image.new('RGB',(canvas,canvas),(255,255,255)); d=ImageDraw.Draw(base)
    font,lines=_layout(d,text,req_size,canvas,margin)
    asc,desc=font.getmetrics(); lh=int((asc+desc)*1.15); y0=max(margin,(canvas-lh*len(lines))//2)
    for i,ln in enumerate(lines):
        bb=d.textbbox((0,0),ln,font=font); x=(canvas-(bb[2]-bb[0]))//2
        jx=int(rng.integers(-10,10)) if rng is not None else 0
        d.text((max(margin,x+jx),y0+i*lh-bb[1]),ln,fill=(0,0,0),font=font)
    return base
def canny_control(base, low=50, high=150, dilate=1):
    gray=np.array(base.convert('L')); edges=cv2.Canny(gray, low, high)
    if dilate>0: edges=cv2.dilate(edges, np.ones((2,2),np.uint8), dilate)
    return Image.fromarray(cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB))
def pil_stone(size,rng):
    t=rng.uniform(0.45,0.70,(size,size))
    for sc in (4,8,16,32):
        hs=max(1,size//sc); n=rng.uniform(-0.06,0.06,(hs,hs))
        ni=Image.fromarray(((n+0.5)*255).clip(0,255).astype(np.uint8)).resize((size,size),Image.NEAREST)
        t+=(np.array(ni).astype(float)/255-0.5)*0.04
    g=(t*int(rng.integers(140,195))).clip(0,255).astype(np.uint8)
    return Image.fromarray(np.stack([g,g,(g*0.95).astype(np.uint8)],2))
print('Рендер/сэмплер/Canny готовы.')

Рендер/сэмплер/Canny готовы.


In [5]:
# ── ЯЧЕЙКА 4. ПАЙПЛАЙН SD3 + ControlNet Canny ──────────────────────────
POSITIVE=("ancient runic inscription carved into dark grey granite, deep incised runes, "
          "weathered rough eroded surface, even diffuse daylight, monochrome grey, "
          "archaeological macro photo, sharp focus, realistic stone texture, no fantasy, no glow")
NEGATIVE=("smooth polished, modern font, fantasy, glow, blur, watermark, extra symbols, "
          "duplicated runes, cartoon, bright colors, illustration, text overlay")
def load_pipe(token):
    if not torch.cuda.is_available(): return None,'pil_only'
    try:
        from diffusers import SD3ControlNetModel, StableDiffusion3ControlNetPipeline
        dtype=torch.float16
        cn=SD3ControlNetModel.from_pretrained("InstantX/SD3-Controlnet-Canny",
                                              torch_dtype=dtype, token=token)
        p=StableDiffusion3ControlNetPipeline.from_pretrained(
            "stabilityai/stable-diffusion-3-medium-diffusers", controlnet=cn,
            torch_dtype=dtype, text_encoder_3=None, tokenizer_3=None, token=token)
        p.enable_model_cpu_offload()
        return p,'sd3_canny'
    except Exception as e:
        print('SD3 не загрузился → PIL fallback:',e); return None,'pil_only'

def generate_one(text, pipe, mode, rng, idx, cfg):
    base=render_runes(text, cfg['canvas'], cfg['req_size'], cfg['margin'], rng)
    if mode=='sd3_canny':
        ctrl=canny_control(base, cfg['canny_low'], cfg['canny_high'], cfg['canny_dilate'])
        g=torch.Generator('cuda').manual_seed(cfg['seed']+idx)
        return pipe(prompt=POSITIVE, negative_prompt=NEGATIVE, control_image=ctrl,
                    num_inference_steps=cfg['steps'], guidance_scale=cfg['cfg_scale'],
                    controlnet_conditioning_scale=cfg['controlnet_scale'],
                    generator=g, width=cfg['canvas'], height=cfg['canvas']).images[0]
    stone=pil_stone(cfg['canvas'], rng); m=(np.array(base.convert('L'))>200).astype(float)
    return Image.fromarray((np.array(stone).astype(float)*m[...,None]
                            +np.array(base).astype(float)*(1-m[...,None])).clip(0,255).astype(np.uint8))
print('Пайплайн готов.')

Пайплайн готов.


In [ ]:
# ── ЯЧЕЙКА 5. ЗАПУСК: PNG → zip-батчи по 50 (всё в /kaggle/working) ─────
CFG=dict(n_samples=2000, canvas=1024, req_size=140, margin=44,
         min_words=1, max_words=4,
         steps=28, cfg_scale=7.0,
         controlnet_scale=0.65,
         canny_low=50, canny_high=150, canny_dilate=1,
         temperature=0.7, seed=8888,
         batch_size=50, delete_local_after_zip=True)

WORKER_ID=0
# CFG['seed']=int(np.random.SeedSequence(1001).spawn(4)[WORKER_ID].generate_state(1)[0])

# В Kaggle всё в /kaggle/working: и временные PNG, и финальные zip-батчи (зипуем рядом).
TMP_IMG = WORK/'_tmp_images'; TMP_IMG.mkdir(parents=True, exist_ok=True)
OUT     = WORK/'dataset'/f'shard_{WORKER_ID}'; OUT.mkdir(parents=True, exist_ok=True)
LABELS  = OUT/'labels_all.csv'

class BatchZipper:
    """Складывает каждые batch_size образцов в один zip внутри /kaggle/working."""
    def __init__(self, img_dir, out_dir, batch_size, delete_after):
        self.img_dir=Path(img_dir); self.out_dir=Path(out_dir)
        self.batch_size=batch_size; self.delete_after=delete_after
        self.pending=[]; self.batch_idx=0
        ex=sorted(self.out_dir.glob('batch_*.zip'))
        if ex: self.batch_idx=int(ex[-1].stem.split('_')[1])+1
    def add(self, rec):
        self.pending.append(rec)
        if len(self.pending)>=self.batch_size: self.flush()
    def flush(self):
        if not self.pending: return
        self.out_dir.mkdir(parents=True, exist_ok=True)
        zip_path=self.out_dir/f'batch_{self.batch_idx:04d}.zip'
        with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
            for r in self.pending:
                p=self.img_dir/r['filename']
                if p.exists(): zf.write(p, f"images/{r['filename']}")
            zf.writestr(f"labels_{self.batch_idx:04d}.csv",
                        pd.DataFrame(self.pending).to_csv(index=False))
        if self.delete_after:
            for r in self.pending:
                p=self.img_dir/r['filename']
                if p.exists(): p.unlink()
        size_mb=zip_path.stat().st_size/1e6
        print(f'  batch_{self.batch_idx:04d}.zip готов ({len(self.pending)} шт, {size_mb:.1f} MB)')
        self.batch_idx+=1; self.pending.clear()

sampler=CorpusSampler(corpus, temperature=CFG['temperature'])
pipe,mode=load_pipe(HF_TOKEN); print('Режим:',mode)
rng=np.random.default_rng(CFG['seed'])

records,start=[],0
if LABELS.exists():
    try:
        ex=pd.read_csv(LABELS); start=len(ex); records=ex.to_dict('records')
        for _ in range(start): sampler.phrase(rng,CFG['min_words'],CFG['max_words'])
        print('Продолжение с',start)
    except Exception: start,records=0,[]

zipper=BatchZipper(TMP_IMG, OUT, CFG['batch_size'], CFG['delete_local_after_zip'])
def safe(t): return t.replace('/','-').replace('\\','-').replace(' ','_')

try:
    for i in tqdm(range(start,CFG['n_samples']), initial=start, total=CFG['n_samples']):
        runic,translit=sampler.phrase(rng,CFG['min_words'],CFG['max_words'])
        fn=f"syn_w{WORKER_ID}_{i:06d}_{safe(translit)}_{safe(runic)}.png"
        try: img=generate_one(runic, pipe, mode, rng, i, CFG)
        except RuntimeError as e:
            if 'out of memory' in str(e).lower(): torch.cuda.empty_cache(); gc.collect(); continue
            raise
        img.save(TMP_IMG/fn)
        rec={'filename':fn,'runic':runic,'translit':translit}
        records.append(rec); zipper.add(rec)
        if (i+1)%CFG['batch_size']==0:
            pd.DataFrame(records).to_csv(LABELS, index=False, encoding='utf-8')
finally:
    zipper.flush()
    pd.DataFrame(records).to_csv(LABELS, index=False, encoding='utf-8')
    print(f'\nГотово: {len(records)} образцов → {OUT} ({zipper.batch_idx} zip-батчей)')
    print('Скачать результаты: правая панель Output → Download (или save kernel и подключайте как Dataset).')

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

...

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]